<a href="https://colab.research.google.com/github/morozovsolncev/ontology_of_synthesis/blob/main/matrix_5_(1_4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import networkx as nx

# =============================================================================
# ЭКСПЕРИМЕНТ: ПОРОГ КАТАЛИЗА В ЗАВИСИМОСТИ ОТ ε_A
# =============================================================================
# Фиксируем долю A = 0.005 (0.5%), варьируем ε_A от 0 до 1.0
# Смотрим, при каком ε_A катализ перестаёт работать (frac_B_in < 0.9)
# =============================================================================

print("=" * 80)
print("ЭКСПЕРИМЕНТ: ПОРОГ КАТАЛИЗА ПРИ 0.5% ДОЛИ A")
print("=" * 80)
print()

dim = 3
sigma = 4.0
percentile_threshold = 90
N_total = 600
frac_A = 0.005  # фиксируем 0.5%
N_A = int(N_total * frac_A)  # = 3
N_B = N_total - N_A

epsilon_B = 10.0
epsilon_A_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

print("ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:")
print(f"  dim = {dim}")
print(f"  N_total = {N_total}")
print(f"  sigma = {sigma}")
print(f"  percentile_threshold = {percentile_threshold}%")
print(f"  epsilon_B = {epsilon_B} (тёмная материя)")
print(f"  frac_A (фиксировано) = {frac_A} (N_A = {N_A})")
print(f"  epsilon_A_list = {epsilon_A_list}")
print()

# -----------------------------------------------------------------------------
# ФУНКЦИИ
# -----------------------------------------------------------------------------
def generate_random_hermitian(dim):
    real_part = np.random.randn(dim, dim)
    real_part = (real_part + real_part.T) / 2
    imag_part = np.random.randn(dim, dim)
    imag_part = (imag_part - imag_part.T) / 2
    return real_part + 1j * imag_part

def generate_non_hermitian(dim, epsilon):
    H = generate_random_hermitian(dim)
    A = generate_random_hermitian(dim)
    return H + 1j * epsilon * A

def commutator_norm(H1, H2):
    if H1.shape != H2.shape:
        return 1e10
    comm = H1 @ H2 - H2 @ H1
    return np.linalg.norm(comm, ord='fro')

def resonance_probability(H1, H2, sigma):
    return np.exp(-commutator_norm(H1, H2)**2 / sigma**2)

# -----------------------------------------------------------------------------
# ОСНОВНОЙ ЦИКЛ
# -----------------------------------------------------------------------------
print("-" * 80)
print(f"{'eps_A':>8} | {'size_main':>10} | {'frac_B_in':>10} | {'max_imag_eig':>12}")
print("-" * 80)

for eps_A in epsilon_A_list:
    np.random.seed(42)

    matrices = []
    types = []

    for _ in range(N_A):
        matrices.append(generate_non_hermitian(dim, eps_A))
        types.append('A')

    for _ in range(N_B):
        matrices.append(generate_non_hermitian(dim, epsilon_B))
        types.append('B')

    N = len(matrices)

    prob_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            p = resonance_probability(matrices[i], matrices[j], sigma)
            prob_matrix[i, j] = p
            prob_matrix[j, i] = p

    all_probs = prob_matrix[np.triu_indices(N, k=1)]
    threshold = np.percentile(all_probs, percentile_threshold)

    G = nx.Graph()
    G.add_nodes_from(range(N))
    for i in range(N):
        for j in range(i+1, N):
            if prob_matrix[i, j] > threshold:
                G.add_edge(i, j)

    components = list(nx.connected_components(G))
    main_cluster = max(components, key=len) if components else set()
    size_main = len(main_cluster)

    count_B_in = sum(1 for v in main_cluster if types[v] == 'B')
    frac_B_in = count_B_in / N_B if N_B > 0 else 0

    if size_main >= 10:
        cluster_matrices = [matrices[v] for v in main_cluster]
        avg_matrix = np.mean(cluster_matrices, axis=0)
        eigenvals = np.linalg.eigvals(avg_matrix)
        max_imag = np.max(np.abs(np.imag(eigenvals)))
    else:
        max_imag = None

    print(f"{eps_A:8.2f} | {size_main:10d} | {frac_B_in:10.3f} | {max_imag:12.2e}")

print("-" * 80)

print("\n" + "=" * 80)
print("ВЫВОДЫ: МАКСИМАЛЬНЫЙ ε_A ДЛЯ ЭФФЕКТИВНОГО КАТАЛИЗА ПРИ 0.5% ДОЛИ")
print("=" * 80)
print("(данные в таблице — определи визуально, при каком eps_A frac_B_in падает ниже 0.9)")
print("=" * 80)
print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
print("=" * 80)

ЭКСПЕРИМЕНТ: ПОРОГ КАТАЛИЗА ПРИ 0.5% ДОЛИ A

ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:
  dim = 3
  N_total = 600
  sigma = 4.0
  percentile_threshold = 90%
  epsilon_B = 10.0 (тёмная материя)
  frac_A (фиксировано) = 0.005 (N_A = 3)
  epsilon_A_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

--------------------------------------------------------------------------------
   eps_A |  size_main |  frac_B_in | max_imag_eig
--------------------------------------------------------------------------------
    0.00 |        600 |      1.000 |     9.46e-01
    0.10 |        600 |      1.000 |     9.46e-01
    0.20 |        600 |      1.000 |     9.47e-01
    0.30 |        600 |      1.000 |     9.47e-01
    0.40 |        600 |      1.000 |     9.47e-01
    0.50 |        599 |      0.998 |     9.74e-01
    0.60 |        599 |      0.998 |     9.74e-01
    0.70 |        597 |      0.995 |     1.02e+00
    0.80 |        593 |      0.988 |     1.02e+00
    0.90 |        588 